In [1]:
import boto3
import pandas as pd
import numpy as np
import os
import pickle
from pandas.api.types import is_numeric_dtype
from pprint import pprint
import matplotlib.pyplot as plt
import datetime as dt

In [2]:
print(f'Latest run date: {dt.datetime.now()}')

Latest run date: 2024-03-15 15:58:28.441876


### Functions

In [3]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

### Constants

In [4]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# date column
str_datecol = 'applicationdate__app'
# output
str_dirname_output = './output'

Project: 20231010-gen-xii


### Create output directory

In [5]:
# create dir
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    print(f'Directory {str_dirname_output} already exists')

Directory ./output already exists


### Import data for approval/decline model

In [6]:
%%time

str_filename = 'df_raw.gzip'
str_uri = f's3://{str_project}/01_ad/01_data_prep/01_data_collection/output/{str_filename}'
df = pd.read_parquet(str_uri)

# show
df

CPU times: user 2min 13s, sys: 1min 13s, total: 3min 27s
Wall time: 14.6 s


,bitdebtor__base,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,dtmfunded__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,...,us934d__tu,us934s__tu,us935b__tu,us935c__tu,us935d__tu,us935s__tu,score_bankcard__tu,score_cvpropensity__tu,finscore__tu,intscore__ln
0,0.0,20181103.0,20181103.0,NaN,NaN,201810.0,73140.0,2018-11-03,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,20181103.0,NaN,20181103.0,NaN,201810.0,72817.0,2018-11-03,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,20181103.0,NaN,20181103.0,NaN,201810.0,74494.0,2018-11-03,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,20181030.0,NaN,20181030.0,NaN,201810.0,61357.0,2018-10-30,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,20181031.0,20181031.0,NaN,NaN,201810.0,61417.0,2018-10-31,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2100693,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,NaN,555.0,588.0
2100694,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,451.0,571.0,574.0
2100695,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,447.0,616.0,568.0
2100696,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,501.0,648.0,595.0


In [7]:
%%time

# drop features that are in the target data
list_cols = [
    'target', # --> bitdefault__app
    'applicationdate__app', # --> dtmstampcreation__app
    'fltapprovedpayment__app', # --> payment__app
    'inttype__app', # --> intopenbktype
    'fltapprovedpricewholesale__app', # --> bookvalue__app
    'fltamountfinanced__app', # --> amtfinanced__app
    'bigaccountid__app',
    'bigdebtorid__app',
    'bigaccountid',
    'bigdebtorid',
    'bitdebtor__base',
    'bitdebtor__tu',
    'bitdebtor__app',
    'dtmfunded__base',
    'dtmfunded__tu',
    'dtmfunded__app',
    'fltamountfinanced__app',
    'intterm__app',
    'vehicleyear__app',
    'bitnew__app',
    'vehiclemake__app',
    'fltdowncash__app',
    'fltapproveddowntotal__app',
    'pti__app',
    'bitservicecontract__app',
    'fltadvance__app',
]
df.drop(list_cols, axis=1, inplace=True)

# show
df

CPU times: user 5.55 s, sys: 11.7 s, total: 17.3 s
Wall time: 13 s


,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,us934d__tu,us934s__tu,us935b__tu,us935c__tu,us935d__tu,us935s__tu,score_bankcard__tu,score_cvpropensity__tu,finscore__tu,intscore__ln
0,20181103.0,20181103.0,NaN,201810.0,73140.0,2018-11-03,0.0,AU,2018-11-12,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20181103.0,NaN,20181103.0,201810.0,72817.0,2018-11-03,0.0,AU,2018-11-10,17188.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20181103.0,NaN,20181103.0,201810.0,74494.0,2018-11-03,0.0,AU,2018-11-06,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20181030.0,NaN,20181030.0,201810.0,61357.0,2018-10-30,0.0,AU,2018-11-15,13065.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20181031.0,20181031.0,NaN,201810.0,61417.0,2018-10-31,0.0,AU,2018-10-31,20150.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2100693,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,NaN,555.0,588.0
2100694,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,451.0,571.0,574.0
2100695,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,447.0,616.0,568.0
2100696,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,501.0,648.0,595.0


### Subset to funded accounts only

In [8]:
str_col = 'bitfunded__app'
ser_val_counts = df[str_col].value_counts()
# show
ser_val_counts

bitfunded__app
0.0    1770866
1.0     186089
Name: count, dtype: int64

In [9]:
# propna
flt_propna = df[str_col].isnull().mean()
print(f'Proportion NaN in {str_col}: {flt_propna:0.3f}')

Proportion NaN in bitfunded__app: 0.068


In [10]:
%%time

df = df[df[str_col] == 1]

# show
df

CPU times: user 3.09 s, sys: 3.33 s, total: 6.42 s
Wall time: 3.73 s


,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,us934d__tu,us934s__tu,us935b__tu,us935c__tu,us935d__tu,us935s__tu,score_bankcard__tu,score_cvpropensity__tu,finscore__tu,intscore__ln
10,20181107.0,20181107.0,NaN,201810.0,95513.0,2018-11-07,1.0,AU,2018-11-06,17496.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,20181121.0,20181121.0,NaN,201810.0,109430.0,2018-11-21,1.0,AU,2018-11-21,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,20181117.0,20181227.0,NaN,201810.0,108781.0,2018-11-17,1.0,AU,2018-11-23,15069.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,20181004.0,20181004.0,NaN,201810.0,2090.0,2018-10-04,1.0,AU,2018-10-31,12333.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47,20181020.0,20181020.0,NaN,201810.0,53961.0,2018-10-20,1.0,AU,2018-10-25,2331.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2100299,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,-3.0,-3.0,-3.0,-3.0,-3.0,-3.0,5.0,604.0,668.0,606.0
2100369,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,0.0,571.0,570.0,571.0
2100427,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,-3.0,-3.0,-3.0,-3.0,-3.0,-3.0,0.0,530.0,555.0,556.0
2100538,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,NaN,482.0,559.0


### Create target

In [11]:
%%time

# import target
str_filename = 'df_target_pd.csv'
str_uri = f's3://{str_project}/ad_hoc/target_pd/{str_filename}'
df_target = pd.read_csv(str_uri)

# subset to MOB >= 72
#df_target = df_target[df_target['monthonbooks__app'] >= 72]

# convert to datetime
df_target['dtmstampcreation__app'] = pd.to_datetime(df_target['dtmstampcreation__app'])
df_target['dealerstampcreation__app'] = pd.to_datetime(df_target['dealerstampcreation__app'])

# rename
dict_rename = {
    'bitdefault__app': 'target',
    'dtmstampcreation__app': 'applicationdate__app',
#     'payment__app': 'fltapprovedpayment__app',
#     'intopenbktype__app': 'inttype__app',
#     'bookvalue__app': 'fltapprovedpricewholesale__app',
#     'amtfinanced__app': 'fltamountfinanced__app',
}
df_target.rename(columns=dict_rename, inplace=True)

# show
df_target

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:273: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


CPU times: user 913 ms, sys: 129 ms, total: 1.04 s
Wall time: 2.85 s


<timed exec>:4: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.


,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,applicationdate__app,dtmfunded__app,target,monthonbooks__app,runningnetloss__app,amtfinanced__app,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
0,133767817566601,1337678,1756660,1,2013-10-01 10:24:32.527,2013-10-21 00:00:00.000,0,72,0.0,12152.00,...,1500.0,1500.0,314.19,0.222419,0.049405,0,0.940365,auto,0,2006-04-02 17:40:05.000
1,133774717567541,1337747,1756754,1,2013-10-01 11:01:41.130,2013-10-11 00:00:00.000,0,72,0.0,15831.73,...,500.0,500.0,364.53,0.510817,0.115263,0,1.048115,suv,1,2010-09-28 11:06:49.813
2,133778017567981,1337780,1756798,1,2013-10-01 11:25:23.807,2013-10-04 00:00:00.000,0,72,0.0,17335.26,...,1040.0,1040.0,394.88,0.396283,0.121472,0,1.121117,auto,0,2011-08-31 09:02:45.850
3,133780717568341,1337807,1756834,1,2013-10-01 11:41:51.333,2013-10-14 00:00:00.000,0,72,0.0,15422.00,...,500.0,500.0,413.68,0.505432,0.158861,0,1.094022,auto,0,2012-01-26 09:55:34.480
4,133781117568381,1337811,1756838,1,2013-10-01 11:44:35.337,2013-10-10 00:00:00.000,0,72,0.0,18570.70,...,1000.0,1000.0,450.11,0.482546,0.082613,1,1.146076,auto,1,2006-01-04 10:58:20.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196834,481138661285821,4811386,6128583,0,2019-12-31 07:56:11.937,2020-02-18 00:00:00.000,0,44,0.0,22631.56,...,1000.0,1000.0,527.34,0.371485,0.053058,0,1.058707,suv,0,2018-11-21 11:01:46.307
196835,481150161287221,4811501,6128723,0,2019-12-31 09:10:49.117,2020-01-08 00:00:00.000,0,45,0.0,20024.75,...,500.0,500.0,454.07,0.300602,0.063171,1,1.234773,suv,0,2009-06-11 17:08:59.937
196836,481157161288091,4811571,6128810,0,2019-12-31 09:43:20.773,2020-01-31 00:00:00.000,0,45,0.0,15796.96,...,0.0,0.0,344.86,0.280712,0.100332,1,1.221195,auto,1,2019-08-09 13:53:46.860
196837,481178461290801,4811784,6129081,0,2019-12-31 11:14:46.693,2020-01-09 00:00:00.000,0,45,0.0,21666.29,...,1000.0,1000.0,443.03,0.271795,0.068680,0,1.106953,auto,1,2015-01-21 14:44:01.150


### Join

In [12]:
df = pd.merge(
    left=df,
    right=df_target,
    left_on='uniqueid',
    right_on='uniqueid__app',
    how='inner',
)

# show
df

,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
0,20181103.0,20181103.0,NaN,201810.0,69614.0,2018-11-03,1.0,AU,2018-11-05,6847.0,...,755.01,755.01,258.37,0.327746,0.039116,0,0.910195,auto,0,2018-02-16 12:20:23.657
1,20181103.0,20181103.0,NaN,201810.0,69614.0,2018-11-03,1.0,AU,2018-11-05,6847.0,...,755.01,755.01,258.37,0.327746,0.039116,0,0.910195,auto,0,2018-02-16 12:20:23.657
2,20181103.0,20181103.0,NaN,201810.0,74719.0,2018-11-03,1.0,AU,2018-11-03,27323.0,...,4750.00,-96.00,735.23,0.486804,0.111285,0,1.049339,suv,1,2007-02-15 09:05:41.530
3,20181103.0,20181103.0,NaN,201810.0,74719.0,2018-11-03,1.0,AU,2018-11-03,27323.0,...,4750.00,-96.00,735.23,0.486804,0.111285,0,1.049339,suv,1,2007-02-15 09:05:41.530
4,20181030.0,20181030.0,NaN,201810.0,65604.0,2018-10-30,1.0,AU,2018-10-30,0.0,...,0.00,1000.00,429.94,0.191788,0.071203,0,1.082675,truck,1,2011-05-20 14:53:37.283
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186170,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,0.00,1150.00,505.00,0.466477,0.082986,0,0.976902,auto,1,2007-07-09 17:18:19.440
186171,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,0.00,1150.00,505.00,0.466477,0.082986,0,0.976902,auto,1,2007-07-09 17:18:19.440
186172,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,500.00,500.00,472.10,0.352323,0.121456,0,1.149966,auto,0,2013-02-21 14:58:51.357
186173,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,12000.00,16751.82,776.23,0.299975,0.062550,0,0.741723,suv,0,2011-09-20 11:51:00.783


### Remove duplicate unique id

In [13]:
df.drop_duplicates(subset=['uniqueid'], keep='last', inplace=True)
df

,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
1,20181103.0,20181103.0,NaN,201810.0,69614.0,2018-11-03,1.0,AU,2018-11-05,6847.0,...,755.01,755.01,258.37,0.327746,0.039116,0,0.910195,auto,0,2018-02-16 12:20:23.657
3,20181103.0,20181103.0,NaN,201810.0,74719.0,2018-11-03,1.0,AU,2018-11-03,27323.0,...,4750.00,-96.00,735.23,0.486804,0.111285,0,1.049339,suv,1,2007-02-15 09:05:41.530
4,20181030.0,20181030.0,NaN,201810.0,65604.0,2018-10-30,1.0,AU,2018-10-30,0.0,...,0.00,1000.00,429.94,0.191788,0.071203,0,1.082675,truck,1,2011-05-20 14:53:37.283
6,20181108.0,20181108.0,NaN,201810.0,80146.0,2018-11-08,1.0,AU,2018-11-23,0.0,...,0.00,0.00,449.88,0.445813,0.083992,0,1.049787,van,1,2012-01-10 16:05:32.420
7,20181110.0,20181110.0,NaN,201810.0,86665.0,2018-11-10,1.0,AU,2018-11-10,11110.0,...,500.00,500.00,446.83,0.424512,0.083581,1,1.242675,auto,0,2006-09-13 11:22:48.827
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186167,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,0.00,0.00,465.96,0.392829,0.089291,1,1.145127,auto,1,2010-04-28 13:41:49.647
186169,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,0.00,0.00,290.71,0.419962,0.109622,0,1.249284,auto,0,2011-06-21 16:14:20.870
186171,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,0.00,1150.00,505.00,0.466477,0.082986,0,0.976902,auto,1,2007-07-09 17:18:19.440
186172,NaN,NaN,NaN,NaN,NaN,nan,NaN,nan,nan,NaN,...,500.00,500.00,472.10,0.352323,0.121456,0,1.149966,auto,0,2013-02-21 14:58:51.357


### Write to s3

In [14]:
%%time

# to parquet
str_filename = 'df_raw.gzip'
str_uri = f's3://{str_project}/09_early_indicators/10_loss_forecasting/01_get_data/{str_filename}'
df.to_parquet(
    path=str_uri,
    compression='gzip',
)

CPU times: user 56 s, sys: 263 ms, total: 56.3 s
Wall time: 56.3 s
